# Trigger a Power Automate Flow from Python

**Goal:** Send an HTTP POST request from Python to fire a live Power Automate flow and inspect the response — in under 2 minutes.

**Estimated time:** 90 seconds

**Prerequisite:** You have created an HTTP-triggered instant cloud flow in Power Automate and copied its HTTP POST URL. If you have not done this yet, follow the setup steps in [Module 01: Your First Cloud Flow](../modules/module_01_first_cloud_flow/) before continuing.

---

## How HTTP-Triggered Flows Work

The **"When an HTTP request is received"** connector (Premium) exposes a public HTTPS endpoint that Power Automate generates for your flow. Any system — a Python script, a CI/CD pipeline, a webhook — can fire the flow by sending an HTTP POST to that URL.

```
Python script
    │
    │  POST https://prod-XX.westeurope.logic.azure.com/workflows/.../triggers/manual/run
    │  Body: {"source": "python", "message": "hello"}
    │
    ▼
Power Automate flow
    │
    ├── Parse JSON body
    ├── Do work (send email, write to SharePoint, post to Teams…)
    └── Return HTTP response (optional)
```

The URL contains a shared-access signature (SAS token) in the query string, so **no separate authentication header is required** — the URL itself is the credential. Treat it like a password.

---

## Set Your Flow URL

Paste the HTTP POST URL from the Power Automate portal into the cell below.

To retrieve it:
1. Open your flow at https://make.powerautomate.com
2. Click the **"When an HTTP request is received"** trigger card
3. Copy the value from **"HTTP POST URL"**

The URL looks like:
```
https://prod-XX.REGION.logic.azure.com:443/workflows/GUID/triggers/manual/paths/invoke?api-version=2016-06-01&sp=...&sv=...&sig=...
```

In [ ]:
# Paste your HTTP POST URL here.
# The URL already contains the SAS token — no separate auth header is needed.
FLOW_HTTP_POST_URL = "https://prod-XX.REGION.logic.azure.com:443/workflows/YOUR_FLOW_URL_HERE"

# Validate that the URL has been replaced before making the request.
if "YOUR_FLOW_URL_HERE" in FLOW_HTTP_POST_URL:
    print("ACTION REQUIRED: Replace FLOW_HTTP_POST_URL with your actual flow URL before running.")
else:
    print(f"Flow URL set. Ready to trigger.")
    print(f"  Endpoint: {FLOW_HTTP_POST_URL[:80]}...")

## Trigger the Flow — Minimal Request

The simplest valid request sends an empty JSON body. Power Automate returns `202 Accepted` immediately and runs the flow asynchronously — the response arrives before the flow finishes executing.

In [ ]:
import requests
import json
from datetime import datetime, timezone

# Guard: skip the live call if the URL has not been set.
if "YOUR_FLOW_URL_HERE" in FLOW_HTTP_POST_URL:
    print("Skipping live request — FLOW_HTTP_POST_URL not set.")
    print("Set the URL above and re-run this cell.")
else:
    # An empty JSON body {} is valid for flows with no required input schema.
    response = requests.post(
        FLOW_HTTP_POST_URL,
        headers={"Content-Type": "application/json"},
        json={},          # Empty body — flow uses defaults for all inputs
        timeout=15
    )

    print(f"Status : {response.status_code} {response.reason}")
    print(f"Headers: {dict(response.headers)}")
    print()

    # 202 Accepted is the expected success code for async flow execution.
    # The response body is empty — the flow runs in the background.
    if response.status_code == 202:
        print("Flow triggered successfully.")
        print("Check your flow's run history at https://make.powerautomate.com to see the execution.")
    elif response.status_code == 200:
        # Some flows are configured to return a response immediately (synchronous pattern).
        print("Flow returned a synchronous response:")
        try:
            print(json.dumps(response.json(), indent=2))
        except Exception:
            print(response.text)
    elif response.status_code == 400:
        print("Bad request — the flow expects a specific JSON schema in the body.")
        print("See the next cell for how to pass input parameters.")
    elif response.status_code in (401, 403):
        print("Auth error — the SAS token in your URL may be expired or invalid.")
        print("Regenerate the URL from the flow's trigger card.")
    else:
        print(f"Unexpected status. Response body: {response.text[:500]}")

## Pass Input Parameters via JSON Body

When a flow defines an **input schema** on its HTTP trigger, it expects specific fields in the JSON body. You define the schema in the portal by clicking **"Use sample payload to generate schema"** in the trigger card and pasting an example JSON object.

Once the schema is defined, any field you include in the body becomes available as dynamic content inside the flow — you can reference it in actions the same way you reference trigger outputs.

The example below shows a flow that expects three fields: `source`, `message`, and `priority`.

In [ ]:
import requests
import json
from datetime import datetime, timezone

# Build a structured payload matching your flow's input schema.
# Modify these fields to match whatever schema your flow defines.
payload = {
    "source": "python-quick-start",           # Identifies where the trigger came from
    "message": "Triggered from Jupyter notebook",
    "priority": "normal",                      # Custom field the flow can branch on
    "timestamp": datetime.now(timezone.utc).isoformat()  # ISO 8601 timestamp
}

print("Payload to send:")
print(json.dumps(payload, indent=2))
print()

if "YOUR_FLOW_URL_HERE" in FLOW_HTTP_POST_URL:
    print("Skipping live request — FLOW_HTTP_POST_URL not set.")
else:
    response = requests.post(
        FLOW_HTTP_POST_URL,
        headers={"Content-Type": "application/json"},
        json=payload,
        timeout=15
    )

    print(f"Status : {response.status_code} {response.reason}")

    if response.status_code == 202:
        print("Flow triggered with parameters.")
        print("The flow received your payload as trigger outputs and is now executing.")
    elif response.status_code == 200:
        print("Synchronous response from flow:")
        try:
            result = response.json()
            print(json.dumps(result, indent=2))
        except Exception:
            print(response.text)
    else:
        print(f"Status {response.status_code}: {response.text[:300]}")

## Reading a Synchronous Response

By default, Power Automate runs flows asynchronously and returns `202 Accepted` immediately. To receive a result back in the same HTTP call, add a **"Response"** action to the end of your flow. When a Response action is present, Power Automate waits (up to 120 seconds) for the flow to finish and returns `200 OK` with whatever body you configure in that action.

This pattern is called a **"request-response flow"** and is common when Python code needs to read data that the flow fetched — for example, querying a SharePoint list and returning the results as JSON.

The parse logic below handles both cases automatically.

In [ ]:
def trigger_flow_and_parse(url: str, payload: dict, timeout: int = 30) -> dict:
    """
    Trigger a Power Automate HTTP-triggered flow and return a structured result.

    Handles both the asynchronous (202) and synchronous (200) response patterns.

    Parameters
    ----------
    url     : The HTTP POST URL from the flow's trigger card (includes SAS token)
    payload : Dict of input parameters matching the flow's JSON schema
    timeout : Seconds to wait before giving up (use >120 only for long-running flows)

    Returns
    -------
    Dict with keys: success (bool), status_code (int), body (any), headers (dict)
    """
    try:
        response = requests.post(
            url,
            headers={"Content-Type": "application/json"},
            json=payload,
            timeout=timeout
        )

        # Parse response body if present
        body = None
        if response.content:
            try:
                body = response.json()
            except ValueError:
                # Non-JSON response body (plain text, HTML error page, etc.)
                body = response.text

        return {
            "success": response.status_code in (200, 202),
            "status_code": response.status_code,
            "async_mode": response.status_code == 202,  # True = flow still running
            "body": body,
            "headers": dict(response.headers)
        }

    except requests.exceptions.Timeout:
        return {"success": False, "status_code": None, "body": "Request timed out", "headers": {}}
    except requests.exceptions.ConnectionError as exc:
        return {"success": False, "status_code": None, "body": str(exc), "headers": {}}


# Demonstrate the helper (live call skipped if URL is not set)
if "YOUR_FLOW_URL_HERE" not in FLOW_HTTP_POST_URL:
    result = trigger_flow_and_parse(
        FLOW_HTTP_POST_URL,
        {"source": "python-quick-start", "message": "test via helper"}
    )
    print(f"Success   : {result['success']}")
    print(f"Status    : {result['status_code']}")
    print(f"Async mode: {result['async_mode']} (True = 202 Accepted, flow still running)")
    if result["body"]:
        print(f"Body      : {json.dumps(result['body'], indent=2)}")
else:
    print("trigger_flow_and_parse() defined — set FLOW_HTTP_POST_URL to run a live call.")
    print()
    # Show what a successful async result looks like
    example_result = {
        "success": True,
        "status_code": 202,
        "async_mode": True,
        "body": None,
        "headers": {"x-ms-workflow-run-id": "08585...", "Date": "Sun, 08 Mar 2026 12:00:00 GMT"}
    }
    print("Example result for a 202 Accepted response:")
    print(json.dumps(example_result, indent=2))

## Next Steps

You know how to trigger a Power Automate flow from Python and handle both response patterns.

| Notebook | What You Will Do |
|----------|------------------|
| **[03_graph_api_basics.ipynb](03_graph_api_basics.ipynb)** | Authenticate with MSAL and call the Microsoft Graph API |
| **[Module 01: Your First Cloud Flow](../modules/module_01_first_cloud_flow/)** | Build an HTTP-triggered flow end-to-end in the portal |
| **[Module 02: Triggers and Connectors](../modules/module_02_triggers_connectors/)** | Deep dive into all trigger types and the connector library |